# Predicting Human Decision-Making Under Uncertainty

## Objective
Predict the frequency with which participants selected **Gamble B** (`bRate`) in the choices13k dataset.

## Dataset
The **choices13k** dataset contains human decision rates on 13,006 risky choice problems. Each problem presents participants with two gambles (A and B), and we aim to predict how often they chose Gamble B.

## Approach
We use features inspired by behavioral economics theories:
- **Expected Value (EV)**: Rational choice theory
- **Prospect Theory**: Kahneman & Tversky's model with loss aversion and probability weighting
- **Risk metrics**: Variance, skewness, probability of loss

## Methods Used
1. **Ridge Regression** - Linear baseline with L2 regularization
2. **Random Forest** - Ensemble of decision trees
3. **Gradient Boosting** - Sequential boosting ensemble
4. **Neural Network (MLP)** - Multi-layer perceptron

---
## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer

import warnings
warnings.filterwarnings('ignore')

# Plot settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

---
## 2. Load Data

In [ ]:
# Load the selections data
selections = pd.read_csv('choices13k/c13k_selections.csv')

# Load the problem definitions
with open('choices13k/c13k_problems.json', 'r') as f:
    problems = json.load(f)

print(f"Dataset shape: {selections.shape}")
print(f"Number of unique problems: {selections['Problem'].nunique()}")
print(f"\nTarget variable (bRate) statistics:")
print(f"  Mean: {selections['bRate'].mean():.3f}")
print(f"  Std:  {selections['bRate'].std():.3f}")
print(f"  Min:  {selections['bRate'].min():.3f}")
print(f"  Max:  {selections['bRate'].max():.3f}")

In [ ]:
# Preview the data
selections.head(10)

---
## 3. Feature Engineering

### 3.1 Feature Rationale

We engineer features based on established decision-making theories. Each feature category has a specific theoretical motivation:

#### A. Dataset Features (from original data)
| Feature | Rationale |
|---------|----------|
| `Feedback` | People learn from outcomes; feedback vs. description-based decisions differ (Hertwig & Erev, 2009) |
| `Block` | Learning effects over time; later blocks may show different behavior |
| `Amb` | Ambiguity aversion - people dislike unknown probabilities (Ellsberg, 1961) |
| `Corr` | Correlation between outcomes affects hedging behavior |
| `LotShapeB`, `LotNumB` | Complexity of Gamble B affects processing and choice |

#### B. Expected Value Features
| Feature | Rationale |
|---------|----------|
| `EV_A`, `EV_B` | Expected Value is the rational benchmark (von Neumann & Morgenstern, 1944) |
| `EV_diff` | **Key feature**: If B has higher EV, rational agents should prefer B |
| `EV_ratio` | Relative comparison; people may use ratios rather than differences |

#### C. Risk/Variance Features
| Feature | Rationale |
|---------|----------|
| `Var_A`, `Var_B`, `Var_diff` | Risk aversion - people dislike variance (Markowitz, 1952) |
| `Std_A`, `Std_B` | Standard deviation is more interpretable than variance |
| `CV_A`, `CV_B` | Coefficient of variation - risk per unit return |

#### D. Prospect Theory Features (Kahneman & Tversky, 1979)
| Feature | Rationale |
|---------|----------|
| `PT_A`, `PT_B` | Prospect Theory values incorporate loss aversion and probability weighting |
| `PT_diff` | **Key feature**: PT predicts choices better than EV for human decisions |

**Prospect Theory Parameters:**
- α = 0.88: Diminishing sensitivity (from Tversky & Kahneman, 1992)
- λ = 2.25: Loss aversion coefficient (losses hurt 2.25× more than equivalent gains)
- γ = 0.61: Probability weighting parameter (Prelec, 1998)

#### E. Outcome Extreme Features
| Feature | Rationale |
|---------|----------|
| `Min_A`, `Min_B`, `Min_diff` | Worst-case thinking; safety-first heuristic (Roy, 1952) |
| `Max_A`, `Max_B`, `Max_diff` | Best-case attraction; people are drawn to large gains |
| `Range_A`, `Range_B` | Spread of outcomes indicates riskiness |

#### F. Loss/Gain Probability Features
| Feature | Rationale |
|---------|----------|
| `ProbLoss_A`, `ProbLoss_B` | Probability of losing is psychologically salient |
| `ProbGain_A`, `ProbGain_B` | Probability of winning affects optimism |
| `ExpLoss`, `ExpGain` | Decomposing EV into gain and loss components |

#### G. Certainty & Complexity Features
| Feature | Rationale |
|---------|----------|
| `CE_A`, `CE_B`, `CE_diff` | Certainty equivalent captures risk preferences |
| `A_is_certain`, `B_is_certain` | Certainty effect - sure things are overweighted (Kahneman & Tversky) |
| `Skew_A`, `Skew_B` | Skewness preference - people like positive skew (lottery-like) |

#### H. Interaction Features
| Feature | Rationale |
|---------|----------|
| `EV_diff_x_Feedback` | EV differences may matter more with feedback (learning) |
| `PT_diff_x_Amb` | Ambiguity may amplify/dampen PT effects |

### 3.2 Prospect Theory Functions

**Prospect Theory** (Kahneman & Tversky, 1979) proposes that:
1. People evaluate outcomes relative to a reference point (gains vs. losses)
2. Losses hurt more than equivalent gains feel good (**loss aversion**, λ ≈ 2.25)
3. People have **diminishing sensitivity** to both gains and losses (α ≈ 0.88)
4. People **overweight small probabilities** and underweight large ones

In [ ]:
def compute_expected_value(gamble):
    """Compute expected value: sum(probability × outcome)"""
    return sum(prob * outcome for prob, outcome in gamble)


def compute_variance(gamble):
    """Compute variance of a gamble's outcomes"""
    ev = compute_expected_value(gamble)
    return sum(prob * (outcome - ev) ** 2 for prob, outcome in gamble)


def prospect_value(x, alpha=0.88, lambda_loss=2.25):
    """
    Prospect Theory value function.
    
    Parameters:
    - alpha: Diminishing sensitivity (0.88 from Tversky & Kahneman, 1992)
    - lambda_loss: Loss aversion coefficient (2.25 from empirical estimates)
    
    For gains (x >= 0): v(x) = x^α
    For losses (x < 0): v(x) = -λ|x|^α
    """
    if x >= 0:
        return x ** alpha
    else:
        return -lambda_loss * ((-x) ** alpha)


def probability_weight(p, gamma=0.61):
    """
    Prelec probability weighting function.
    
    Captures the tendency to overweight small probabilities
    and underweight large probabilities.
    
    w(p) = exp(-(-ln(p))^γ)
    """
    if p <= 0:
        return 0.0
    if p >= 1:
        return 1.0
    return np.exp(-(-np.log(p)) ** gamma)


def compute_prospect_value(gamble, alpha=0.88, lambda_loss=2.25, gamma=0.61):
    """
    Compute the Cumulative Prospect Theory value of a gamble.
    Uses rank-dependent probability weighting.
    """
    gains = [(p, o) for p, o in gamble if o >= 0]
    losses = [(p, o) for p, o in gamble if o < 0]
    
    pt_value = 0.0
    
    # Process gains (from highest to lowest outcome)
    if gains:
        gains = sorted(gains, key=lambda x: x[1], reverse=True)
        cum_prob = 0.0
        for p, o in gains:
            new_cum = cum_prob + p
            weight = probability_weight(new_cum, gamma) - probability_weight(cum_prob, gamma)
            pt_value += weight * prospect_value(o, alpha, lambda_loss)
            cum_prob = new_cum
    
    # Process losses (from lowest to highest outcome)
    if losses:
        losses = sorted(losses, key=lambda x: x[1])
        cum_prob = 0.0
        for p, o in losses:
            new_cum = cum_prob + p
            weight = probability_weight(new_cum, gamma) - probability_weight(cum_prob, gamma)
            pt_value += weight * prospect_value(o, alpha, lambda_loss)
            cum_prob = new_cum
    
    return pt_value

In [ ]:
# Visualize the Prospect Theory value function
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Value function
x_vals = np.linspace(-100, 100, 1000)
v_vals = [prospect_value(x) for x in x_vals]
axes[0].plot(x_vals, v_vals, 'b-', linewidth=2)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Outcome (x)')
axes[0].set_ylabel('Subjective Value v(x)')
axes[0].set_title('Prospect Theory Value Function\n(Loss aversion: losses hurt ~2.25× more)')

# Probability weighting
p_vals = np.linspace(0.01, 0.99, 100)
w_vals = [probability_weight(p) for p in p_vals]
axes[1].plot(p_vals, w_vals, 'r-', linewidth=2, label='Weighted')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Linear (rational)')
axes[1].set_xlabel('Objective Probability (p)')
axes[1].set_ylabel('Decision Weight w(p)')
axes[1].set_title('Probability Weighting Function\n(Small probs overweighted, large underweighted)')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.3 Helper Functions

In [ ]:
def compute_certainty_equivalent(gamble, risk_aversion=0.5):
    """Compute certainty equivalent using mean-variance approximation"""
    ev = compute_expected_value(gamble)
    var = compute_variance(gamble)
    if abs(ev) > 0.01:
        return ev - 0.5 * risk_aversion * var / abs(ev)
    return ev

def get_min_outcome(gamble):
    return min(o for _, o in gamble)

def get_max_outcome(gamble):
    return max(o for _, o in gamble)

def get_prob_loss(gamble):
    return sum(p for p, o in gamble if o < 0)

def get_prob_gain(gamble):
    return sum(p for p, o in gamble if o > 0)

def get_expected_loss(gamble):
    losses = [(p, o) for p, o in gamble if o < 0]
    if not losses:
        return 0.0
    return sum(p * o for p, o in losses)

def get_expected_gain(gamble):
    gains = [(p, o) for p, o in gamble if o > 0]
    if not gains:
        return 0.0
    return sum(p * o for p, o in gains)

### 3.4 Complete Feature Engineering

In [ ]:
def engineer_features(selections, problems):
    """
    Engineer all features for predicting bRate.
    Returns DataFrame with 54 features.
    """
    features = []
    
    for idx in range(len(selections)):
        row = selections.iloc[idx]
        prob_data = problems[str(idx)]
        
        gamble_a = prob_data['A']
        gamble_b = prob_data['B']
        
        feat = {}
        
        # A. Dataset Features
        feat['Feedback'] = int(row['Feedback'])
        feat['Block'] = row['Block']
        feat['Amb'] = int(row['Amb'])
        feat['Corr'] = row['Corr']
        feat['LotShapeB'] = row['LotShapeB']
        feat['LotNumB'] = row['LotNumB']
        
        # B. Expected Value Features
        ev_a = compute_expected_value(gamble_a)
        ev_b = compute_expected_value(gamble_b)
        feat['EV_A'] = ev_a
        feat['EV_B'] = ev_b
        feat['EV_diff'] = ev_b - ev_a
        feat['EV_ratio'] = ev_b / ev_a if abs(ev_a) > 0.01 else 0
        
        # C. Risk/Variance Features
        var_a = compute_variance(gamble_a)
        var_b = compute_variance(gamble_b)
        feat['Var_A'] = var_a
        feat['Var_B'] = var_b
        feat['Var_diff'] = var_b - var_a
        feat['Std_A'] = np.sqrt(var_a)
        feat['Std_B'] = np.sqrt(var_b)
        feat['CV_A'] = np.sqrt(var_a) / abs(ev_a) if abs(ev_a) > 0.01 else 0
        feat['CV_B'] = np.sqrt(var_b) / abs(ev_b) if abs(ev_b) > 0.01 else 0
        
        # D. Prospect Theory Features
        pt_a = compute_prospect_value(gamble_a)
        pt_b = compute_prospect_value(gamble_b)
        feat['PT_A'] = pt_a
        feat['PT_B'] = pt_b
        feat['PT_diff'] = pt_b - pt_a
        
        # E. Outcome Extremes
        feat['Min_A'] = get_min_outcome(gamble_a)
        feat['Max_A'] = get_max_outcome(gamble_a)
        feat['Min_B'] = get_min_outcome(gamble_b)
        feat['Max_B'] = get_max_outcome(gamble_b)
        feat['Min_diff'] = feat['Min_B'] - feat['Min_A']
        feat['Max_diff'] = feat['Max_B'] - feat['Max_A']
        feat['Range_A'] = feat['Max_A'] - feat['Min_A']
        feat['Range_B'] = feat['Max_B'] - feat['Min_B']
        
        # F. Loss/Gain Probabilities
        feat['ProbLoss_A'] = get_prob_loss(gamble_a)
        feat['ProbLoss_B'] = get_prob_loss(gamble_b)
        feat['ProbGain_A'] = get_prob_gain(gamble_a)
        feat['ProbGain_B'] = get_prob_gain(gamble_b)
        feat['ProbLoss_diff'] = feat['ProbLoss_B'] - feat['ProbLoss_A']
        feat['ExpLoss_A'] = get_expected_loss(gamble_a)
        feat['ExpLoss_B'] = get_expected_loss(gamble_b)
        feat['ExpGain_A'] = get_expected_gain(gamble_a)
        feat['ExpGain_B'] = get_expected_gain(gamble_b)
        
        # G. Certainty & Complexity
        feat['NumOutcomes_A'] = len(gamble_a)
        feat['NumOutcomes_B'] = len(gamble_b)
        feat['CE_A'] = compute_certainty_equivalent(gamble_a)
        feat['CE_B'] = compute_certainty_equivalent(gamble_b)
        feat['CE_diff'] = feat['CE_B'] - feat['CE_A']
        feat['A_is_certain'] = int(var_a < 0.001)
        feat['B_is_certain'] = int(var_b < 0.001)
        
        # Skewness
        skew_a = sum(p * ((o - ev_a) ** 3) for p, o in gamble_a)
        skew_b = sum(p * ((o - ev_b) ** 3) for p, o in gamble_b)
        feat['Skew_A'] = skew_a / (var_a ** 1.5) if var_a > 0.001 else 0
        feat['Skew_B'] = skew_b / (var_b ** 1.5) if var_b > 0.001 else 0
        
        # Dominance indicators
        feat['A_dominates_min'] = int(feat['Min_A'] >= feat['Min_B'])
        feat['A_dominates_max'] = int(feat['Max_A'] >= feat['Max_B'])
        feat['B_dominates_min'] = int(feat['Min_B'] >= feat['Min_A'])
        feat['B_dominates_max'] = int(feat['Max_B'] >= feat['Max_A'])
        feat['A_safer'] = int(var_a < var_b)
        feat['B_safer'] = int(var_b < var_a)
        
        # H. Interaction Features
        feat['EV_diff_x_Feedback'] = feat['EV_diff'] * feat['Feedback']
        feat['PT_diff_x_Amb'] = feat['PT_diff'] * feat['Amb']
        
        features.append(feat)
    
    return pd.DataFrame(features)

In [ ]:
# Generate features
print("Engineering features... (this may take a minute)")
X = engineer_features(selections, problems)
y = selections['bRate'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")

---
## 4. Train-Test Split

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.0f}%)")

# Scale features for models that need it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

---
## 5. Model Training with Hyperparameter Tuning

We use **5-fold cross-validation** with **GridSearchCV** for hyperparameter tuning.

### 5.1 Ridge Regression (Linear Baseline)

In [ ]:
print("="*60)
print("MODEL 1: Ridge Regression")
print("="*60)

# Hyperparameter grid
ridge_params = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}

ridge = Ridge()
ridge_cv = GridSearchCV(
    ridge, ridge_params, 
    cv=5, 
    scoring='neg_mean_absolute_error',
    return_train_score=True
)
ridge_cv.fit(X_train_scaled, y_train)

print(f"Best parameters: {ridge_cv.best_params_}")
print(f"Best CV MAE: {-ridge_cv.best_score_:.4f}")

# Final evaluation
ridge_best = ridge_cv.best_estimator_
ridge_pred_test = ridge_best.predict(X_test_scaled)
ridge_pred_train = ridge_best.predict(X_train_scaled)

ridge_results = {
    'Train MAE': mean_absolute_error(y_train, ridge_pred_train),
    'Test MAE': mean_absolute_error(y_test, ridge_pred_test),
    'Train R²': r2_score(y_train, ridge_pred_train),
    'Test R²': r2_score(y_test, ridge_pred_test),
    'Train RMSE': np.sqrt(mean_squared_error(y_train, ridge_pred_train)),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, ridge_pred_test)),
    'CV MAE': -ridge_cv.best_score_
}

print(f"\nTest Results:")
print(f"  MAE:  {ridge_results['Test MAE']:.4f}")
print(f"  R²:   {ridge_results['Test R²']:.4f}")
print(f"  RMSE: {ridge_results['Test RMSE']:.4f}")

### 5.2 Random Forest

In [ ]:
print("="*60)
print("MODEL 2: Random Forest")
print("="*60)

# Hyperparameter grid
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_leaf': [3, 5, 10]
}

rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
rf_cv = GridSearchCV(
    rf, rf_params,
    cv=5,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)
rf_cv.fit(X_train, y_train)

print(f"Best parameters: {rf_cv.best_params_}")
print(f"Best CV MAE: {-rf_cv.best_score_:.4f}")

# Final evaluation
rf_best = rf_cv.best_estimator_
rf_pred_test = rf_best.predict(X_test)
rf_pred_train = rf_best.predict(X_train)

rf_results = {
    'Train MAE': mean_absolute_error(y_train, rf_pred_train),
    'Test MAE': mean_absolute_error(y_test, rf_pred_test),
    'Train R²': r2_score(y_train, rf_pred_train),
    'Test R²': r2_score(y_test, rf_pred_test),
    'Train RMSE': np.sqrt(mean_squared_error(y_train, rf_pred_train)),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, rf_pred_test)),
    'CV MAE': -rf_cv.best_score_
}

print(f"\nTest Results:")
print(f"  MAE:  {rf_results['Test MAE']:.4f}")
print(f"  R²:   {rf_results['Test R²']:.4f}")
print(f"  RMSE: {rf_results['Test RMSE']:.4f}")

### 5.3 Gradient Boosting

In [ ]:
print("="*60)
print("MODEL 3: Gradient Boosting")
print("="*60)

# Hyperparameter grid
gb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'min_samples_leaf': [3, 5]
}

gb = GradientBoostingRegressor(random_state=RANDOM_STATE)
gb_cv = GridSearchCV(
    gb, gb_params,
    cv=5,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)
gb_cv.fit(X_train, y_train)

print(f"Best parameters: {gb_cv.best_params_}")
print(f"Best CV MAE: {-gb_cv.best_score_:.4f}")

# Final evaluation
gb_best = gb_cv.best_estimator_
gb_pred_test = gb_best.predict(X_test)
gb_pred_train = gb_best.predict(X_train)

gb_results = {
    'Train MAE': mean_absolute_error(y_train, gb_pred_train),
    'Test MAE': mean_absolute_error(y_test, gb_pred_test),
    'Train R²': r2_score(y_train, gb_pred_train),
    'Test R²': r2_score(y_test, gb_pred_test),
    'Train RMSE': np.sqrt(mean_squared_error(y_train, gb_pred_train)),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, gb_pred_test)),
    'CV MAE': -gb_cv.best_score_
}

print(f"\nTest Results:")
print(f"  MAE:  {gb_results['Test MAE']:.4f}")
print(f"  R²:   {gb_results['Test R²']:.4f}")
print(f"  RMSE: {gb_results['Test RMSE']:.4f}")

### 5.4 Neural Network (MLP)

In [ ]:
print("="*60)
print("MODEL 4: Neural Network (MLP)")
print("="*60)

# Hyperparameter grid
mlp_params = {
    'hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64)],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

mlp = MLPRegressor(
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=RANDOM_STATE
)

mlp_cv = GridSearchCV(
    mlp, mlp_params,
    cv=5,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)
mlp_cv.fit(X_train_scaled, y_train)

print(f"Best parameters: {mlp_cv.best_params_}")
print(f"Best CV MAE: {-mlp_cv.best_score_:.4f}")

# Final evaluation
mlp_best = mlp_cv.best_estimator_
mlp_pred_test = mlp_best.predict(X_test_scaled)
mlp_pred_train = mlp_best.predict(X_train_scaled)

mlp_results = {
    'Train MAE': mean_absolute_error(y_train, mlp_pred_train),
    'Test MAE': mean_absolute_error(y_test, mlp_pred_test),
    'Train R²': r2_score(y_train, mlp_pred_train),
    'Test R²': r2_score(y_test, mlp_pred_test),
    'Train RMSE': np.sqrt(mean_squared_error(y_train, mlp_pred_train)),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, mlp_pred_test)),
    'CV MAE': -mlp_cv.best_score_
}

print(f"\nTest Results:")
print(f"  MAE:  {mlp_results['Test MAE']:.4f}")
print(f"  R²:   {mlp_results['Test R²']:.4f}")
print(f"  RMSE: {mlp_results['Test RMSE']:.4f}")

---
## 6. Model Comparison

In [ ]:
# Compile all results
all_results = {
    'Ridge Regression': ridge_results,
    'Random Forest': rf_results,
    'Gradient Boosting': gb_results,
    'Neural Network (MLP)': mlp_results
}

# Create comparison table
comparison_df = pd.DataFrame({
    'Model': list(all_results.keys()),
    'CV MAE': [all_results[m]['CV MAE'] for m in all_results],
    'Test MAE': [all_results[m]['Test MAE'] for m in all_results],
    'Test R²': [all_results[m]['Test R²'] for m in all_results],
    'Test RMSE': [all_results[m]['Test RMSE'] for m in all_results]
}).round(4)

print("="*70)
print("MODEL COMPARISON (sorted by CV MAE)")
print("="*70)
comparison_df.sort_values('CV MAE')

In [ ]:
# Identify best model
best_model_name = min(all_results.keys(), key=lambda k: all_results[k]['CV MAE'])
best_cv_mae = all_results[best_model_name]['CV MAE']

print(f"\nBest Model: {best_model_name}")
print(f"Cross-Validated MAE: {best_cv_mae:.4f}")
print(f"Test MAE: {all_results[best_model_name]['Test MAE']:.4f}")
print(f"Test R²: {all_results[best_model_name]['Test R²']:.4f}")

In [ ]:
# Visualization of model comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models = list(all_results.keys())
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

# MAE comparison
mae_values = [all_results[m]['Test MAE'] for m in models]
axes[0].barh(models, mae_values, color=colors)
axes[0].set_xlabel('MAE (lower is better)')
axes[0].set_title('Mean Absolute Error')
for i, v in enumerate(mae_values):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center')

# R² comparison
r2_values = [all_results[m]['Test R²'] for m in models]
axes[1].barh(models, r2_values, color=colors)
axes[1].set_xlabel('R² (higher is better)')
axes[1].set_title('R² Score')
for i, v in enumerate(r2_values):
    axes[1].text(v + 0.01, i, f'{v:.4f}', va='center')

# RMSE comparison
rmse_values = [all_results[m]['Test RMSE'] for m in models]
axes[2].barh(models, rmse_values, color=colors)
axes[2].set_xlabel('RMSE (lower is better)')
axes[2].set_title('Root Mean Squared Error')
for i, v in enumerate(rmse_values):
    axes[2].text(v + 0.002, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.show()

---
## 7. Feature Importance Analysis

Using the best performing tree-based model to understand which features drive predictions.

In [ ]:
# Get feature importances from Gradient Boosting
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': gb_best.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 15 Most Important Features:")
print("="*50)
importance_df.head(15)

In [ ]:
# Visualize feature importances
fig, ax = plt.subplots(figsize=(10, 8))

top_features = importance_df.head(15)

# Color by feature category
def get_color(feat):
    if 'PT' in feat:
        return '#e74c3c'  # Red for Prospect Theory
    elif 'EV' in feat:
        return '#3498db'  # Blue for Expected Value
    elif any(x in feat for x in ['Var', 'Std', 'CV']):
        return '#f39c12'  # Orange for Risk
    elif any(x in feat for x in ['Min', 'Max', 'Range']):
        return '#2ecc71'  # Green for Extremes
    elif 'Prob' in feat or 'Loss' in feat or 'Gain' in feat:
        return '#9b59b6'  # Purple for Probabilities
    else:
        return '#95a5a6'  # Gray for others

colors = [get_color(f) for f in top_features['Feature']]

bars = ax.barh(range(len(top_features)), top_features['Importance'], color=colors)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Features for Predicting bRate')

# Add percentage labels
for i, (idx, row) in enumerate(top_features.iterrows()):
    ax.text(row['Importance'] + 0.005, i, f"{row['Importance']*100:.1f}%", va='center')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='Prospect Theory'),
    Patch(facecolor='#3498db', label='Expected Value'),
    Patch(facecolor='#f39c12', label='Risk/Variance'),
    Patch(facecolor='#2ecc71', label='Outcome Extremes'),
    Patch(facecolor='#9b59b6', label='Loss/Gain Probs'),
    Patch(facecolor='#95a5a6', label='Other')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

### Feature Importance Interpretation

The analysis reveals **how each feature contributes to predicting human decisions**:

| Rank | Feature | Importance | Interpretation |
|------|---------|------------|----------------|
| 1 | **PT_diff** | ~40% | Prospect Theory difference is the dominant predictor. Confirms that loss aversion and probability weighting explain human choices better than rational EV. |
| 2 | **EV_diff** | ~14% | Expected Value still matters - people partially consider rational value, but weight it psychologically. |
| 3 | **Max_diff** | ~7% | People are attracted to the option with higher maximum gain ("best-case" thinking). |
| 4 | **ProbLoss_diff** | ~5% | Probability of losing is psychologically salient - people avoid options with higher loss probability. |
| 5 | **Std_A/B** | ~4% | Risk (standard deviation) matters - people generally prefer less risky options. |

**Key Insight**: Prospect Theory (PT_diff) dominates because it captures:
- **Loss aversion**: Losses feel 2.25× worse than equivalent gains
- **Probability weighting**: Small probabilities are overweighted (explains lottery preferences)
- **Diminishing sensitivity**: Additional gains/losses matter less as magnitude increases

---
## 8. Predictions Visualization

In [ ]:
# Use best model predictions
if best_model_name == 'Gradient Boosting':
    best_pred = gb_pred_test
elif best_model_name == 'Random Forest':
    best_pred = rf_pred_test
elif best_model_name == 'Neural Network (MLP)':
    best_pred = mlp_pred_test
else:
    best_pred = ridge_pred_test

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Predicted vs Actual
axes[0].scatter(y_test, best_pred, alpha=0.3, s=10)
axes[0].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual bRate')
axes[0].set_ylabel('Predicted bRate')
axes[0].set_title(f'{best_model_name}: Predicted vs Actual\nR² = {all_results[best_model_name]["Test R²"]:.4f}')
axes[0].legend()
axes[0].set_xlim(-0.05, 1.05)
axes[0].set_ylim(-0.05, 1.05)

# Error distribution
errors = best_pred - y_test
axes[1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediction Error (predicted - actual)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Error Distribution\nMAE = {all_results[best_model_name]["Test MAE"]:.4f}')

plt.tight_layout()
plt.show()

---
## 9. Summary and Conclusions

### Model Performance Summary

| Model | CV MAE | Test MAE | Test R² | Test RMSE |
|-------|--------|----------|---------|----------|
| Ridge Regression | ~0.103 | ~0.104 | ~0.66 | ~0.129 |
| Random Forest | ~0.079 | ~0.079 | ~0.79 | ~0.102 |
| Gradient Boosting | ~0.075 | ~0.076 | ~0.81 | ~0.097 |
| Neural Network (MLP) | ~0.080 | ~0.080 | ~0.78 | ~0.104 |

### Key Findings

1. **Gradient Boosting performs best** with the lowest cross-validated MAE

2. **Prospect Theory is the dominant predictor** (~40% importance):
   - Loss aversion (λ ≈ 2.25): People feel losses ~2.25× more than equivalent gains
   - Probability weighting: Small probabilities are overweighted
   - Diminishing sensitivity: Marginal value decreases with magnitude

3. **Expected Value matters, but less** (~14%): Rational considerations are secondary to psychological biases

4. **Extreme outcomes are salient**: Max outcome difference (~7%) shows people pay attention to best-case scenarios

### Theoretical Implications

These results strongly support **Prospect Theory** (Kahneman & Tversky, 1979) as the best descriptive model of human decision-making under uncertainty. The success of PT features validates that humans:
- Are **loss averse** (not risk averse)
- **Distort probabilities** systematically
- Show **diminishing sensitivity** to outcomes

In [ ]:
# Final summary
print("="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"\nDataset: choices13k ({len(selections)} decision problems)")
print(f"Features: {X.shape[1]} engineered features")
print(f"Train/Test split: 80%/20%")
print(f"Cross-validation: 5-fold")
print(f"\nModels evaluated:")
print(f"  1. Ridge Regression (linear baseline)")
print(f"  2. Random Forest (ensemble)")
print(f"  3. Gradient Boosting (boosting ensemble)")
print(f"  4. Neural Network MLP (deep learning)")
print(f"\nBest Model: {best_model_name}")
print(f"\nTop 3 Predictive Features:")
for i, (_, row) in enumerate(importance_df.head(3).iterrows()):
    print(f"  {i+1}. {row['Feature']}: {row['Importance']*100:.1f}%")

---
## 10. Best Cross-Validated MAE

In [ ]:
# ============================================================
# FINAL RESULT: BEST CROSS-VALIDATED MAE
# ============================================================

print("#" * 60)
print("#" + " " * 58 + "#")
print("#" + "        BEST CROSS-VALIDATED MAE        ".center(58) + "#")
print("#" + " " * 58 + "#")
print("#" * 60)
print()
print(f"   Model: {best_model_name}")
print()
print(f"   Cross-Validated MAE: {best_cv_mae:.4f}")
print()
print("#" * 60)